In [ ]:
import joblib 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# pick which model to explain — comment out the others
TAG = "augmented"
# TAG = "qwen"; # TAG = "llama"
# TAG = "qwen_rewrite"; # TAG = "llama_rewrite"

b = joblib.load(f"xgb_bundle_{TAG}.joblib")
clf, tfidf         = b["clf"], b["tfidf"]
feat_names, hand_names = b["feat_names"], b["hand_names"]
Xte, te_x, te_y    = b["Xte"], b["te_x"], np.array(b["te_y"])
hte, htr, tr_y     = b["hte"], b["htr"], np.array(b["tr_y"])
prob, pred         = np.array(b["prob"]), np.array(b["pred"])
print(f"Loaded {TAG}: {Xte.shape[0]} test notes, {len(feat_names)} features")

# Q2: How does the classifier respond across a feature's range? (PDP + ICE)

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
from xgboost import XGBClassifier

# PDP needs interpretable inputs → model on hand-crafted features only
clf_hand = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.5, reg_alpha=1.0, reg_lambda=2.0,
    eval_metric="logloss", random_state=42).fit(htr.values, tr_y)

# plot the features most likely to carry signal (skip any not present)
wanted = ["avg_sentence_length","vocab_richness","connector_density",
          "flesch","sentence_length_cv","word_count"]
feats = [f for f in wanted if f in hand_names]
idx = [hand_names.index(f) for f in feats]

fig, ax = plt.subplots(2, 3, figsize=(16, 9))
PartialDependenceDisplay.from_estimator(
    clf_hand, htr.values, idx, feature_names=hand_names,
    kind="both", ax=ax.ravel()[:len(idx)], random_state=42)
fig.suptitle(f"PDP/ICE — {TAG}  (y = P(synthetic))")
plt.tight_layout(); plt.savefig(f"q2_pdp_{TAG}.png", dpi=150, bbox_inches="tight"); plt.show()